# 05 — Self-Confidence & Self-Verification

Bu notebook 04'te dondurulan ana cevapları okur; ana cevapları yeniden üretmez.

Her `question_id × model_id` için iki davranışsal güven ailesi üretir:

**Self-confidence**
- `self_confidence_raw`
- `self_confidence`
- `self_confidence_scaled`
- parse/format bayrakları

**Self-verification**
- `verification_raw`
- `verification_choice` (`A/B/C`)
- `verification_p_correct`
- `verification_p_incorrect`
- `verification_p_uncertain`
- `verification_margin`
- `verification_entropy_normalized`
- generated choice / probability argmax agreement

Model yalnız soruyu ve 04'teki kendi frozen cevabını görür. `reference_answer`,
`accepted_aliases` ve `manual_is_correct` bu aşamada kullanılmaz.

## 1. Drive ve bağımlılıklar

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip -q install "transformers==4.57.6" "accelerate>=1.3,<2" "bitsandbytes>=0.45" "huggingface_hub>=0.34"

## 2. Imports ve runtime

In [ ]:
import gc, hashlib, json, math, os, platform, re, time, unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch, transformers, accelerate, bitsandbytes as bnb

from google.colab import userdata
from huggingface_hub import HfApi

EXPECTED_TRANSFORMERS_VERSION = "4.57.6"

if transformers.__version__ != EXPECTED_TRANSFORMERS_VERSION:
    raise RuntimeError(
        f"Beklenen transformers={EXPECTED_TRANSFORMERS_VERSION}; "
        f"mevcut={transformers.__version__}. Runtime'ı yeniden başlatın."
    )

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime gereklidir.")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 3. Paths ve dataset version

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_Confidence_Thesis")

RESULTS_STAGE04_DIR = PROJECT_ROOT / "results" / "stage04"
MODEL_SELECTION_DIR = PROJECT_ROOT / "results" / "model_selection"

STAGE05_ROOT = PROJECT_ROOT / "runs" / "self_assessment"
CONF_CACHE_ROOT = STAGE05_ROOT / "cache" / "self_confidence"
VER_CACHE_ROOT = STAGE05_ROOT / "cache" / "self_verification"
EXPORT_ROOT = STAGE05_ROOT / "exports"
RESULTS_STAGE05_DIR = PROJECT_ROOT / "results" / "stage05"

for p in [CONF_CACHE_ROOT, VER_CACHE_ROOT, EXPORT_ROOT, RESULTS_STAGE05_DIR]:
    p.mkdir(parents=True, exist_ok=True)

VALIDATED_DATASET_VERSION = "validated_v002"

STAGE04_COMPLETION_PATH = (
    RESULTS_STAGE04_DIR /
    f"{VALIDATED_DATASET_VERSION}_stage04_complete.json"
)

SELECTION_MANIFEST_PATH = (
    MODEL_SELECTION_DIR /
    "selected_models_v1.json"
)

## 4. Hash ve JSONL yardımcıları

In [ ]:
def canonical_json(obj):
    return json.dumps(
        obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    )

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def sha256_object(obj):
    return sha256_text(canonical_json(obj))

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()

def read_jsonl(path):
    if not path.exists():
        return []
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def append_jsonl_record(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
        f.flush()
        os.fsync(f.fileno())

def atomic_write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
    os.replace(tmp, path)

## 5. Stage-04 ve model-selection manifestlerini doğrula

In [ ]:
if not STAGE04_COMPLETION_PATH.exists():
    raise FileNotFoundError(STAGE04_COMPLETION_PATH)

with open(STAGE04_COMPLETION_PATH, "r", encoding="utf-8") as f:
    stage04_manifest = json.load(f)

if stage04_manifest["dataset_version"] != VALIDATED_DATASET_VERSION:
    raise RuntimeError("Stage-04 dataset version uyuşmuyor.")

with open(SELECTION_MANIFEST_PATH, "r", encoding="utf-8") as f:
    selection_manifest = json.load(f)

selection_manifest_sha256 = file_sha256(SELECTION_MANIFEST_PATH)
selected_models = selection_manifest["selected_models"]

if len(selected_models) < 3:
    raise RuntimeError("En az üç model bekleniyor.")

display(pd.DataFrame(selected_models)[[
    "model_id", "short_name", "revision", "backend",
    "compute_dtype", "model_dtype"
]])

## 6. 04 frozen export'larını yükle ve SHA doğrula

In [ ]:
stage04_exports_by_model = {
    item["model_id"]: item
    for item in stage04_manifest["model_exports"]
}

stage04_rows_by_model = {}

for spec in selected_models:
    model_id = spec["model_id"]
    artifact = stage04_exports_by_model[model_id]
    export_path = Path(artifact["export_file"])

    if file_sha256(export_path) != artifact["export_sha256"]:
        raise RuntimeError(f"Stage-04 SHA uyuşmuyor: {model_id}")

    rows = read_jsonl(export_path)

    if len(rows) != stage04_manifest["n_questions"]:
        raise RuntimeError(f"Stage-04 row count yanlış: {model_id}")

    stage04_rows_by_model[model_id] = rows
    print(model_id, "| rows =", len(rows), "| SHA OK")

## 7. Prompt contract

In [ ]:
SELF_CONFIDENCE_PROMPT_VERSION = "self_confidence_0_100_tr_v1"
SELF_CONFIDENCE_GENERATION_VERSION = "deterministic_short_v1"
SELF_CONFIDENCE_MAX_NEW_TOKENS = 6

SELF_CONFIDENCE_SYSTEM_PROMPT = (
    "Bir soru-cevap çiftindeki cevabın olgusal doğruluğunu değerlendiriyorsun. "
    "Verilen cevabın doğru olduğuna dair güvenini 0 ile 100 arasında tek bir tam sayı ile bildir. "
    "0 kesinlikle yanlış, 100 kesinlikle doğru anlamına gelir. Sadece sayıyı yaz."
)

SELF_CONFIDENCE_USER_TEMPLATE = (
    "Soru:\n{question}\n\n"
    "Verilen cevap:\n{answer}\n\n"
    "Bu cevabın olgusal olarak doğru olduğuna dair güvenin kaç?"
)

VERIFICATION_PROMPT_VERSION = "self_verification_abc_tr_v1"
VERIFICATION_GENERATION_VERSION = "deterministic_abc_v1"
VERIFICATION_PROBABILITY_VERSION = "abc_next_token_restricted_softmax_v1"
VERIFICATION_MAX_NEW_TOKENS = 4

VERIFICATION_SYSTEM_PROMPT = (
    "Bir soru-cevap çiftindeki cevabın olgusal olarak doğru olup olmadığını değerlendiriyorsun. "
    "Yalnızca A, B veya C yaz."
)

VERIFICATION_USER_TEMPLATE = (
    "Soru:\n{question}\n\n"
    "Verilen cevap:\n{answer}\n\n"
    "Cevap soruyu olgusal olarak doğru yanıtlıyor mu?\n"
    "A = Doğru\n"
    "B = Yanlış\n"
    "C = Emin değilim\n\n"
    "Yalnızca A, B veya C yaz."
)

## 8. Parser yardımcıları

In [ ]:
def parse_self_confidence(raw_text):
    if not isinstance(raw_text, str):
        return {"value": None, "parse_success": 0, "format_valid": 0}

    text = unicodedata.normalize("NFKC", raw_text).strip()
    match = re.search(r"(?<!\d)(100|[0-9]{1,2})(?!\d)", text)

    if not match:
        return {"value": None, "parse_success": 0, "format_valid": 0}

    value = int(match.group(1))
    strict = re.fullmatch(r"\s*(100|[0-9]{1,2})\s*%?\s*\.?\s*", text)

    return {
        "value": value,
        "parse_success": 1,
        "format_valid": int(strict is not None),
    }

def parse_verification_choice(raw_text):
    if not isinstance(raw_text, str):
        return {"choice": None, "parse_success": 0, "format_valid": 0}

    text = unicodedata.normalize("NFKC", raw_text).strip().upper()
    match = re.match(r"^\s*([ABC])(?:\b|[\.\)\]:,-])?", text)
    choice = match.group(1) if match else None
    strict = re.fullmatch(r"\s*[ABC]\s*\.?\s*", text)

    return {
        "choice": choice,
        "parse_success": int(choice is not None),
        "format_valid": int(strict is not None),
    }

## 9. Model loader ve Gemma HF_TOKEN

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoProcessor,
    Gemma3ForConditionalGeneration, BitsAndBytesConfig
)

DTYPE_MAP = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}

def get_dtype(name):
    return DTYPE_MAP[name]

def quant_config(spec):
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=get_dtype(spec["compute_dtype"]),
    )

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def get_tokenizer(frontend):
    return frontend.tokenizer if hasattr(frontend, "tokenizer") else frontend

def get_hf_token_if_needed(spec):
    if spec["backend"] != "gemma3_processor":
        return None

    token = userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Colab Secrets içinde HF_TOKEN bulunamadı.")

    HfApi(token=token).model_info(
        spec["model_id"],
        revision=spec["revision"],
    )
    return token

def load_model(spec):
    clear_gpu()

    model_id = spec["model_id"]
    revision = spec["revision"]
    backend = spec["backend"]
    token = get_hf_token_if_needed(spec)

    print("Model:", model_id)
    print("Revision:", revision)
    print("Backend:", backend)
    print("Compute dtype:", spec["compute_dtype"])

    if backend == "gemma3_processor":
        frontend = AutoProcessor.from_pretrained(
            model_id, revision=revision, token=token, trust_remote_code=False
        )

        model = Gemma3ForConditionalGeneration.from_pretrained(
            model_id,
            revision=revision,
            token=token,
            quantization_config=quant_config(spec),
            device_map="auto",
            torch_dtype=get_dtype(spec["model_dtype"]),
        )
    else:
        frontend = AutoTokenizer.from_pretrained(
            model_id,
            revision=revision,
            trust_remote_code=spec["trust_remote_code"],
            use_fast=True,
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            revision=revision,
            quantization_config=quant_config(spec),
            device_map="auto",
            trust_remote_code=spec["trust_remote_code"],
            torch_dtype=get_dtype(spec["model_dtype"]),
        )

    model.eval()
    tokenizer = get_tokenizer(frontend)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    if model.generation_config.pad_token_id is None:
        model.generation_config.pad_token_id = tokenizer.pad_token_id

    print("Loaded:", model.__class__.__name__)
    print("Tokenizer:", tokenizer.__class__.__name__)

    return frontend, model

## 10. Assessment input helper

In [ ]:
def build_assessment_inputs(
    frontend, model, spec, system_prompt, user_prompt
):
    if spec["backend"] == "gemma3_processor":
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": system_prompt}],
            },
            {
                "role": "user",
                "content": [{"type": "text", "text": user_prompt}],
            },
        ]

        inputs = frontend.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )
    else:
        tokenizer = get_tokenizer(frontend)

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        if spec["disable_thinking"]:
            try:
                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=False,
                )
            except TypeError:
                prompt = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
        else:
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

        inputs = tokenizer(prompt, return_tensors="pt")

    return {k: v.to(model.device) for k, v in inputs.items()}

def decode_generated(frontend, generated_ids):
    tokenizer = get_tokenizer(frontend)
    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

## 11. Self-confidence inference

In [ ]:
@torch.inference_mode()
def infer_self_confidence(frontend, model, spec, question, frozen_answer):
    user_prompt = SELF_CONFIDENCE_USER_TEMPLATE.format(
        question=question,
        answer=frozen_answer,
    )

    inputs = build_assessment_inputs(
        frontend, model, spec,
        SELF_CONFIDENCE_SYSTEM_PROMPT,
        user_prompt,
    )

    prompt_len = inputs["input_ids"].shape[1]

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    generated = model.generate(
        **inputs,
        max_new_tokens=SELF_CONFIDENCE_MAX_NEW_TOKENS,
        do_sample=False,
        return_dict_in_generate=True,
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    raw = decode_generated(
        frontend,
        generated.sequences[0][prompt_len:],
    )

    parsed = parse_self_confidence(raw)
    value = parsed["value"]

    return {
        "self_confidence_raw": raw,
        "self_confidence": value,
        "self_confidence_scaled": (
            float(value / 100.0) if value is not None else None
        ),
        "self_confidence_parse_success": parsed["parse_success"],
        "self_confidence_format_valid": parsed["format_valid"],
        "self_confidence_generation_seconds": float(elapsed),
    }

## 12. Verification probability extractor

A/B/C sınıfları ilk assistant-token pozisyonundaki logits üzerinden değerlendirilir.
Olasılıklar yalnız bu üç sınıf içinde normalize edilir. Bu nedenle
`verification_p_correct` kalibre edilmiş doğruluk olasılığı değil, bir confidence feature'dır.

In [ ]:
def get_verification_label_token_ids(tokenizer):
    result = {}

    for label in ["A", "B", "C"]:
        ids = tokenizer.encode(
            label,
            add_special_tokens=False,
        )

        if len(ids) != 1:
            raise RuntimeError(
                f"{label} tek token değil: {ids}. "
                "Probability extractor bu tokenizer için revize edilmeli."
            )

        result[label] = int(ids[0])

    if len(set(result.values())) != 3:
        raise RuntimeError("A/B/C token ID'leri benzersiz değil.")

    return result

def normalized_three_class_probs(logits, token_ids):
    values = torch.tensor(
        [
            float(logits[token_ids["A"]].item()),
            float(logits[token_ids["B"]].item()),
            float(logits[token_ids["C"]].item()),
        ],
        dtype=torch.float64,
    )

    p = torch.softmax(values, dim=0).cpu().numpy()

    return {
        "A": float(p[0]),
        "B": float(p[1]),
        "C": float(p[2]),
    }

def verification_entropy_normalized(probs):
    v = np.array(
        [probs["A"], probs["B"], probs["C"]],
        dtype=float,
    )

    entropy = -float(
        np.sum(v * np.log(np.clip(v, 1e-15, 1.0)))
    )

    return float(entropy / math.log(3.0))

## 13. Self-verification inference

In [ ]:
@torch.inference_mode()
def infer_self_verification(frontend, model, spec, question, frozen_answer):
    tokenizer = get_tokenizer(frontend)
    label_ids = get_verification_label_token_ids(tokenizer)

    user_prompt = VERIFICATION_USER_TEMPLATE.format(
        question=question,
        answer=frozen_answer,
    )

    inputs = build_assessment_inputs(
        frontend, model, spec,
        VERIFICATION_SYSTEM_PROMPT,
        user_prompt,
    )

    forward_out = model(**inputs)
    next_logits = forward_out.logits[0, -1, :].float()

    probs = normalized_three_class_probs(
        next_logits,
        label_ids,
    )

    pA, pB, pC = probs["A"], probs["B"], probs["C"]

    margin = float(
        pA - max(pB, pC)
    )

    entropy_norm = verification_entropy_normalized(probs)
    argmax_class = max(probs, key=probs.get)

    prompt_len = inputs["input_ids"].shape[1]

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    generated = model.generate(
        **inputs,
        max_new_tokens=VERIFICATION_MAX_NEW_TOKENS,
        do_sample=False,
        return_dict_in_generate=True,
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    raw = decode_generated(
        frontend,
        generated.sequences[0][prompt_len:],
    )

    parsed = parse_verification_choice(raw)
    choice = parsed["choice"]

    agreement = (
        int(choice == argmax_class)
        if choice is not None
        else None
    )

    return {
        "verification_raw": raw,
        "verification_choice": choice,
        "verification_parse_success": parsed["parse_success"],
        "verification_format_valid": parsed["format_valid"],

        "verification_label_token_id_A": label_ids["A"],
        "verification_label_token_id_B": label_ids["B"],
        "verification_label_token_id_C": label_ids["C"],

        "verification_p_correct": pA,
        "verification_p_incorrect": pB,
        "verification_p_uncertain": pC,
        "verification_margin": margin,
        "verification_entropy_normalized": entropy_norm,
        "verification_argmax_class": argmax_class,
        "verification_generation_probability_agreement": agreement,
        "verification_generation_seconds": float(elapsed),
    }

## 14. Task config hash ve cache key

In [ ]:
def confidence_contract(spec):
    return {
        "task": "self_confidence",
        "model_id": spec["model_id"],
        "model_revision": spec["revision"],
        "backend": spec["backend"],
        "compute_dtype": spec["compute_dtype"],
        "model_dtype": spec["model_dtype"],
        "prompt_version": SELF_CONFIDENCE_PROMPT_VERSION,
        "system_prompt_sha256": sha256_text(SELF_CONFIDENCE_SYSTEM_PROMPT),
        "user_template_sha256": sha256_text(SELF_CONFIDENCE_USER_TEMPLATE),
        "generation_version": SELF_CONFIDENCE_GENERATION_VERSION,
        "max_new_tokens": SELF_CONFIDENCE_MAX_NEW_TOKENS,
        "do_sample": False,
    }

def verification_contract(spec):
    return {
        "task": "self_verification",
        "model_id": spec["model_id"],
        "model_revision": spec["revision"],
        "backend": spec["backend"],
        "compute_dtype": spec["compute_dtype"],
        "model_dtype": spec["model_dtype"],
        "prompt_version": VERIFICATION_PROMPT_VERSION,
        "system_prompt_sha256": sha256_text(VERIFICATION_SYSTEM_PROMPT),
        "user_template_sha256": sha256_text(VERIFICATION_USER_TEMPLATE),
        "generation_version": VERIFICATION_GENERATION_VERSION,
        "probability_version": VERIFICATION_PROBABILITY_VERSION,
        "max_new_tokens": VERIFICATION_MAX_NEW_TOKENS,
        "do_sample": False,
    }

def stage05_cache_key(stage04_row, task_hash):
    return sha256_object(
        {
            "stage04_inference_cache_key": stage04_row["inference_cache_key"],
            "frozen_answer_sha256": sha256_text(stage04_row["answer"]),
            "task_config_hash": task_hash,
        }
    )

## 15. Incremental model run

In [ ]:
def run_stage05_for_model(spec):
    model_id = spec["model_id"]
    short_name = spec["short_name"]
    stage04_rows = stage04_rows_by_model[model_id]

    conf_contract = confidence_contract(spec)
    ver_contract = verification_contract(spec)

    conf_hash = sha256_object(conf_contract)
    ver_hash = sha256_object(ver_contract)

    conf_short = conf_hash[:12]
    ver_short = ver_hash[:12]

    conf_dir = CONF_CACHE_ROOT / short_name
    ver_dir = VER_CACHE_ROOT / short_name
    conf_dir.mkdir(parents=True, exist_ok=True)
    ver_dir.mkdir(parents=True, exist_ok=True)

    conf_cache_path = conf_dir / f"{conf_short}.jsonl"
    ver_cache_path = ver_dir / f"{ver_short}.jsonl"

    conf_cache = {
        r["stage05_cache_key"]: r
        for r in read_jsonl(conf_cache_path)
    }

    ver_cache = {
        r["stage05_cache_key"]: r
        for r in read_jsonl(ver_cache_path)
    }

    items = []

    for row in stage04_rows:
        ck = stage05_cache_key(row, conf_hash)
        vk = stage05_cache_key(row, ver_hash)

        items.append(
            {
                "base": row,
                "conf_key": ck,
                "ver_key": vk,
                "need_conf": ck not in conf_cache,
                "need_ver": vk not in ver_cache,
            }
        )

    conf_pending = sum(int(x["need_conf"]) for x in items)
    ver_pending = sum(int(x["need_ver"]) for x in items)

    print("\nMODEL:", model_id)
    print("Rows:", len(items))
    print("Self-confidence new:", conf_pending)
    print("Verification new:", ver_pending)

    peak_gpu_gb = None

    if conf_pending or ver_pending:
        frontend, model = load_model(spec)

        label_ids = get_verification_label_token_ids(
            get_tokenizer(frontend)
        )
        print("Verification label token IDs:", label_ids)

        torch.cuda.reset_peak_memory_stats()

        counter = 0

        for item in items:
            if not (item["need_conf"] or item["need_ver"]):
                continue

            counter += 1
            base = item["base"]
            question = base["question"]
            answer = base["answer"]

            if item["need_conf"]:
                cr = infer_self_confidence(
                    frontend, model, spec, question, answer
                )

                rec = {
                    "stage05_cache_key": item["conf_key"],
                    "task_config_hash": conf_hash,
                    "question_id": base["question_id"],
                    "model_id": model_id,
                    "model_revision": spec["revision"],
                    "stage04_inference_cache_key": base["inference_cache_key"],
                    "frozen_answer_sha256": sha256_text(answer),
                    "created_at_utc": utc_now_iso(),
                    **cr,
                }

                append_jsonl_record(conf_cache_path, rec)
                conf_cache[item["conf_key"]] = rec

            if item["need_ver"]:
                vr = infer_self_verification(
                    frontend, model, spec, question, answer
                )

                rec = {
                    "stage05_cache_key": item["ver_key"],
                    "task_config_hash": ver_hash,
                    "question_id": base["question_id"],
                    "model_id": model_id,
                    "model_revision": spec["revision"],
                    "stage04_inference_cache_key": base["inference_cache_key"],
                    "frozen_answer_sha256": sha256_text(answer),
                    "created_at_utc": utc_now_iso(),
                    **vr,
                }

                append_jsonl_record(ver_cache_path, rec)
                ver_cache[item["ver_key"]] = rec

            c = conf_cache[item["conf_key"]]
            v = ver_cache[item["ver_key"]]

            print(
                f"[{counter:04d}] {base['question_id']} | "
                f"ANS={answer!r} | "
                f"CONF={c['self_confidence']} | "
                f"VER={v['verification_choice']} | "
                f"pA={v['verification_p_correct']:.3f} "
                f"pB={v['verification_p_incorrect']:.3f} "
                f"pC={v['verification_p_uncertain']:.3f}"
            )

        peak_gpu_gb = float(
            torch.cuda.max_memory_allocated() / (1024 ** 3)
        )

        del model, frontend
        clear_gpu()

    else:
        print("Tüm kayıtlar cache'de. Model yüklenmedi.")

    export_dir = EXPORT_ROOT / VALIDATED_DATASET_VERSION
    export_dir.mkdir(parents=True, exist_ok=True)

    conf_export = (
        export_dir /
        f"{short_name}_self_confidence_{conf_short}.jsonl"
    )

    ver_export = (
        export_dir /
        f"{short_name}_self_verification_{ver_short}.jsonl"
    )

    conf_rows, ver_rows = [], []

    for item in items:
        base = item["base"]
        common = {
            "dataset_version": VALIDATED_DATASET_VERSION,
            "question_id": base["question_id"],
            "model_id": model_id,
            "model_revision": spec["revision"],
            "stage04_inference_cache_key": base["inference_cache_key"],
            "frozen_answer_sha256": sha256_text(base["answer"]),
        }

        c = conf_cache[item["conf_key"]]
        v = ver_cache[item["ver_key"]]

        conf_rows.append(
            {
                **common,
                **{
                    k: val
                    for k, val in c.items()
                    if k.startswith("self_confidence")
                },
            }
        )

        ver_rows.append(
            {
                **common,
                **{
                    k: val
                    for k, val in v.items()
                    if k.startswith("verification")
                },
            }
        )

    if not conf_export.exists():
        atomic_write_jsonl(conf_export, conf_rows)

    if not ver_export.exists():
        atomic_write_jsonl(ver_export, ver_rows)

    manifest = {
        "stage": "05_self_confidence_verification",
        "dataset_version": VALIDATED_DATASET_VERSION,
        "model_id": model_id,
        "model_revision": spec["revision"],
        "n": len(items),

        "self_confidence_contract": conf_contract,
        "self_confidence_config_hash": conf_hash,

        "self_verification_contract": ver_contract,
        "self_verification_config_hash": ver_hash,

        "self_confidence_export": str(conf_export),
        "self_confidence_export_sha256": file_sha256(conf_export),

        "self_verification_export": str(ver_export),
        "self_verification_export_sha256": file_sha256(ver_export),

        "new_self_confidence_this_run": conf_pending,
        "new_verification_this_run": ver_pending,
        "peak_gpu_memory_gb_for_new_inference": peak_gpu_gb,
        "created_at_utc": utc_now_iso(),
    }

    manifest_path = (
        export_dir /
        f"{short_name}_stage05_manifest.json"
    )

    if not manifest_path.exists():
        with open(manifest_path, "x", encoding="utf-8") as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)

    print("\nSTAGE-05 MODEL TAMAMLANDI")
    print(conf_export)
    print(ver_export)

    return {
        "model_id": model_id,
        "confidence_export": conf_export,
        "verification_export": ver_export,
        "manifest": manifest_path,
    }

## 16. Model index tablosu

In [ ]:
MODEL_INDEX_TABLE = pd.DataFrame(
    [
        {
            "index": i,
            "model_id": s["model_id"],
            "short_name": s["short_name"],
            "compute_dtype": s["compute_dtype"],
        }
        for i, s in enumerate(selected_models)
    ]
)

display(MODEL_INDEX_TABLE)

## 17. Tek modeli çalıştır

Önce bir modeli çalıştırıp ilk logları kontrol edin. Sonra index'i diğer modellere değiştirin.
Aynı modeli tekrar çalıştırmak cache nedeniyle güvenlidir.

In [ ]:
MODEL_INDEX_TO_RUN = 0

stage05_run_result = run_stage05_for_model(
    selected_models[MODEL_INDEX_TO_RUN]
)

stage05_run_result

## 18. Üç model export'unu birleştir

In [ ]:
all_rows = []
missing = []

export_dir = EXPORT_ROOT / VALIDATED_DATASET_VERSION

for spec in selected_models:
    conf_hash = sha256_object(confidence_contract(spec))[:12]
    ver_hash = sha256_object(verification_contract(spec))[:12]

    conf_path = (
        export_dir /
        f"{spec['short_name']}_self_confidence_{conf_hash}.jsonl"
    )
    ver_path = (
        export_dir /
        f"{spec['short_name']}_self_verification_{ver_hash}.jsonl"
    )

    if not conf_path.exists() or not ver_path.exists():
        missing.append(spec["model_id"])
        continue

    cdf = pd.DataFrame(read_jsonl(conf_path))
    vdf = pd.DataFrame(read_jsonl(ver_path))

    merged = cdf.merge(
        vdf,
        on=[
            "dataset_version",
            "question_id",
            "model_id",
            "model_revision",
            "stage04_inference_cache_key",
            "frozen_answer_sha256",
        ],
        how="inner",
        validate="one_to_one",
    )

    all_rows.extend(merged.to_dict(orient="records"))

if missing:
    raise RuntimeError(f"Eksik modeller: {missing}")

combined05_df = pd.DataFrame(all_rows)

expected_rows = (
    stage04_manifest["n_questions"]
    * len(selected_models)
)

print("Stage-05 rows:", len(combined05_df))
print("Expected:", expected_rows)

if len(combined05_df) != expected_rows:
    raise RuntimeError("Stage-05 coverage eksik.")

if combined05_df.duplicated(
    subset=["question_id", "model_id"]
).any():
    raise RuntimeError("Duplicate question_id × model_id.")

## 19. Audit

In [ ]:
audit = (
    combined05_df
    .groupby("model_id")
    .agg(
        n=("question_id", "count"),
        self_confidence_parse_rate=(
            "self_confidence_parse_success", "mean"
        ),
        self_confidence_strict_format_rate=(
            "self_confidence_format_valid", "mean"
        ),
        mean_self_confidence=(
            "self_confidence_scaled", "mean"
        ),
        verification_parse_rate=(
            "verification_parse_success", "mean"
        ),
        verification_strict_format_rate=(
            "verification_format_valid", "mean"
        ),
        verification_generated_probability_agreement_rate=(
            "verification_generation_probability_agreement", "mean"
        ),
        mean_verification_p_correct=(
            "verification_p_correct", "mean"
        ),
        mean_verification_p_uncertain=(
            "verification_p_uncertain", "mean"
        ),
        mean_verification_entropy_normalized=(
            "verification_entropy_normalized", "mean"
        ),
    )
    .reset_index()
)

display(audit)

AUDIT_PATH = (
    RESULTS_STAGE05_DIR /
    f"{VALIDATED_DATASET_VERSION}_stage05_audit.csv"
)

audit.to_csv(AUDIT_PATH, index=False)
print(AUDIT_PATH)

## 20. Probability sanity check

In [ ]:
prob_cols = [
    "verification_p_correct",
    "verification_p_incorrect",
    "verification_p_uncertain",
]

pm = combined05_df[prob_cols].astype(float)

if not np.allclose(
    pm.sum(axis=1).values,
    np.ones(len(pm)),
    atol=1e-8,
):
    raise RuntimeError("A/B/C probability toplamları 1 değil.")

if ((pm < 0) | (pm > 1)).any().any():
    raise RuntimeError("Verification probability [0,1] dışında.")

ent = pd.to_numeric(
    combined05_df["verification_entropy_normalized"],
    errors="coerce",
)

if ent.isna().any():
    raise RuntimeError("Verification entropy missing.")

if ((ent < -1e-12) | (ent > 1 + 1e-12)).any():
    raise RuntimeError("Normalized entropy [0,1] dışında.")

print("VERIFICATION PROBABILITY SANITY CHECK BAŞARILI")

## 21. Parse, agreement ve format audit

İki problem sınıfını ayırıyoruz.

### Kritik extraction problemi
- self-confidence parse edilemedi
- verification A/B/C parse edilemedi
- generated verification choice ile A/B/C probability argmax uyuşmadı

### Format-compliance problemi
Model sınıfı parse edilebilir biçimde üretmiştir fakat “yalnızca A/B/C yaz” talimatına tam uymamıştır.

Örneğin `A. Cevap doğrudur.` için:

```text
verification_choice = A
verification_parse_success = 1
verification_format_valid = 0
```

Format violation nedeniyle inference yeniden üretilmez; raw çıktı ve format flag'i korunur.

In [ ]:
critical_mask = (
    (combined05_df["self_confidence_parse_success"] == 0)
    | (combined05_df["verification_parse_success"] == 0)
    | (
        combined05_df[
            "verification_generation_probability_agreement"
        ] == 0
    )
)

format_mask = (
    (combined05_df["self_confidence_format_valid"] == 0)
    | (combined05_df["verification_format_valid"] == 0)
)

critical_df = combined05_df.loc[
    critical_mask,
    [
        "question_id",
        "model_id",
        "self_confidence_raw",
        "self_confidence",
        "verification_raw",
        "verification_choice",
        "verification_argmax_class",
        "verification_p_correct",
        "verification_p_incorrect",
        "verification_p_uncertain",
    ],
]

format_df = combined05_df.loc[
    format_mask,
    [
        "question_id",
        "model_id",
        "self_confidence_raw",
        "self_confidence_format_valid",
        "verification_raw",
        "verification_choice",
        "verification_format_valid",
    ],
]

print("KRİTİK extraction problemi:", len(critical_df))
print("Format-compliance problemi:", len(format_df))

format_summary = (
    combined05_df
    .groupby("model_id")
    .agg(
        n=("question_id", "count"),
        self_confidence_format_violations=(
            "self_confidence_format_valid",
            lambda s: int((s == 0).sum()),
        ),
        verification_format_violations=(
            "verification_format_valid",
            lambda s: int((s == 0).sum()),
        ),
    )
    .reset_index()
)

display(format_summary)

if not critical_df.empty:
    display(critical_df.head(100))

if not format_df.empty:
    display(format_df.head(30))

CRITICAL_AUDIT_PATH = (
    RESULTS_STAGE05_DIR
    / f"{VALIDATED_DATASET_VERSION}_stage05_critical_problems.csv"
)

FORMAT_AUDIT_PATH = (
    RESULTS_STAGE05_DIR
    / f"{VALIDATED_DATASET_VERSION}_stage05_format_violations.csv"
)

critical_df.to_csv(CRITICAL_AUDIT_PATH, index=False)
format_df.to_csv(FORMAT_AUDIT_PATH, index=False)

if len(critical_df) > 0:
    raise RuntimeError(
        "Kritik Stage-05 extraction problemi bulundu. "
        "Stage-05'i dondurmayın."
    )

print("KRİTİK EXTRACTION AUDIT BAŞARILI")
print("Format ihlalleri raw model davranışı olarak korunacak.")

## 22. Stage-05 completion manifest

In [ ]:
model_artifacts = []

export_dir = EXPORT_ROOT / VALIDATED_DATASET_VERSION

for spec in selected_models:
    manifest_path = (
        export_dir /
        f"{spec['short_name']}_stage05_manifest.json"
    )

    if not manifest_path.exists():
        raise RuntimeError(
            f"Stage05 manifest eksik: {spec['model_id']}"
        )

    with open(manifest_path, "r", encoding="utf-8") as f:
        model_artifacts.append(json.load(f))

STAGE05_COMPLETION_PATH = (
    RESULTS_STAGE05_DIR /
    f"{VALIDATED_DATASET_VERSION}_stage05_complete.json"
)

completion = {
    "stage": "05_self_confidence_verification",
    "dataset_version": VALIDATED_DATASET_VERSION,

    "stage04_completion_manifest": str(STAGE04_COMPLETION_PATH),
    "stage04_completion_manifest_sha256": file_sha256(
        STAGE04_COMPLETION_PATH
    ),

    "selection_manifest_sha256": selection_manifest_sha256,

    "n_questions": stage04_manifest["n_questions"],
    "n_models": len(selected_models),
    "n_model_answer_rows": len(combined05_df),

    "self_confidence_prompt_version": SELF_CONFIDENCE_PROMPT_VERSION,
    "self_verification_prompt_version": VERIFICATION_PROMPT_VERSION,
    "verification_probability_version": VERIFICATION_PROBABILITY_VERSION,

    "model_artifacts": model_artifacts,
    "audit_file": str(AUDIT_PATH),
    "critical_audit_file": str(CRITICAL_AUDIT_PATH),
    "format_audit_file": str(FORMAT_AUDIT_PATH),
    "critical_extraction_problem_count": int(len(critical_df)),
    "format_violation_count": int(len(format_df)),
    "created_at_utc": utc_now_iso(),
}

if STAGE05_COMPLETION_PATH.exists():
    print("Stage-05 completion zaten mevcut:")
    print(STAGE05_COMPLETION_PATH)
else:
    with open(STAGE05_COMPLETION_PATH, "x", encoding="utf-8") as f:
        json.dump(completion, f, ensure_ascii=False, indent=2)

    print("STAGE 05 DONDURULDU:")
    print(STAGE05_COMPLETION_PATH)

# 05 tamamlanma kriterleri

- 04 frozen exports SHA ile doğrulandı
- Ana cevaplar yeniden üretilmedi
- Self-confidence üç model için üretildi
- 0–100 confidence parse edildi
- Self-verification üç model için üretildi
- A/B/C token ID'leri doğrulandı
- `p_correct`, `p_incorrect`, `p_uncertain` üretildi
- Verification margin ve normalized entropy üretildi
- Generated choice / probability argmax agreement kaydedildi
- 360 `question_id × model_id` coverage tamamlandı
- Probability sanity check geçti
- Kritik parse/agreement sorunları audit edildi
- Format-compliance ihlalleri model davranışı olarak ayrı audit edildi
- `validated_vXXX_stage05_complete.json` oluşturuldu

## Sonraki aşama — 06 Self-Consistency

06 ana deterministic cevabı yeniden üretmeyecek. Stochastic K tekrar üzerinden majority frequency,
unique-answer ratio, sample-answer entropy, deterministic-majority agreement ve unknown/refusal
sample oranları üretilecek. `sample_correct_rate` feature olarak kullanılmayacak.